# Phase 1 — Ingestion

For each of the 5 watchlist companies: find FY23/FY24 10-K accession numbers, fetch + parse + segment + chunk + embed the filing text, and extract XBRL facts. Requires `NEON_DATABASE_URL` (Kaggle Secret) and internet enabled.

Run `00_setup_and_schema.ipynb` first.

In [ ]:
import subprocess, sys, os

REPO_URL = "https://github.com/<your-username>/stance.git"  # TODO: set this
if not os.path.isdir("/kaggle/working/stance"):
    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/stance"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/stance[retrieval]"], check=True)
sys.path.insert(0, "/kaggle/working/stance")

In [ ]:
from stance.db import get_engine
from stance.seed import WATCHLIST, FISCAL_YEARS
from stance.ingest import acquire, parse, segment, chunk, embed, facts

engine = get_engine()
print(f"Ingesting {len(WATCHLIST)} companies x {FISCAL_YEARS}")

## 1a. Acquire — find accession numbers, fetch filing HTML, fetch companyfacts

In [ ]:
from sqlalchemy import text

filings_to_process = []  # (cik, ticker, accession_no, fiscal_year, html)
company_facts_cache = {}

for company in WATCHLIST:
    cik, ticker = company["cik"], company["ticker"]
    print(f"--- {ticker} ---")

    submissions = acquire.fetch_submissions(cik)
    target_filings = acquire.find_10k_accessions(submissions, FISCAL_YEARS)
    print(f"  found {len(target_filings)} 10-K filing(s): {[f['accession_no'] for f in target_filings]}")

    for f in target_filings:
        html = acquire.fetch_filing_document(cik, f["accession_no"], f["primary_doc"])
        filings_to_process.append({**f, "cik": cik, "ticker": ticker, "html": html})

        with engine.begin() as conn:
            conn.execute(text(
                "INSERT INTO filings (accession_no, cik, form_type, fiscal_year, filing_date, primary_doc_url, ingest_status) "
                "VALUES (:accession_no, :cik, :form_type, :fiscal_year, :filing_date, :primary_doc, 'acquired') "
                "ON CONFLICT (accession_no) DO NOTHING"
            ), {**f, "cik": cik})

    company_facts_cache[cik] = acquire.fetch_company_facts(cik)

print(f"\nTotal filings to parse: {len(filings_to_process)}")

## 1b. Parse, segment, chunk, embed

In [ ]:
import uuid

total_chunks_inserted = 0

for f in filings_to_process:
    print(f"Parsing {f['ticker']} {f['accession_no']}...")
    text_content = parse.parse_filing_html(f["html"])
    sections = segment.segment_filing(text_content)
    print(f"  segments found: {[s.item_code for s in sections]}")

    all_chunks = []
    for section in sections:
        section_id = f"sec-{uuid.uuid4().hex[:12]}"
        with engine.begin() as conn:
            conn.execute(text(
                "INSERT INTO sections (section_id, accession_no, item_code, title, text) "
                "VALUES (:sid, :acc, :item, :title, :text) ON CONFLICT (section_id) DO NOTHING"
            ), {"sid": section_id, "acc": f["accession_no"], "item": section.item_code,
                "title": section.title, "text": section.text})

        section_chunks = chunk.chunk_section(section.text)
        for c in section_chunks:
            all_chunks.append({
                "chunk_id": c.chunk_id, "section_id": section_id, "text": c.text,
                "accession_no": f["accession_no"], "cik": f["cik"], "ticker": f["ticker"],
                "fiscal_year": f["fiscal_year"], "item_code": section.item_code,
            })

    if not all_chunks:
        continue
    embeddings = embed.embed_texts([c["text"] for c in all_chunks])
    with engine.begin() as conn:
        for c, vec in zip(all_chunks, embeddings):
            conn.execute(text(
                "INSERT INTO chunks (chunk_id, section_id, accession_no, cik, ticker, fiscal_year, "
                "item_code, text, embedding, embed_model, chunker_version) "
                "VALUES (:chunk_id, :section_id, :accession_no, :cik, :ticker, :fiscal_year, "
                ":item_code, :text, :embedding, :embed_model, :chunker_version) "
                "ON CONFLICT (chunk_id) DO NOTHING"
            ), {**c, "embedding": str(vec), "embed_model": "bge-small-en-v1.5", "chunker_version": "chunk-v1"})
    total_chunks_inserted += len(all_chunks)
    print(f"  inserted {len(all_chunks)} chunks")

print(f"\nTotal chunks inserted: {total_chunks_inserted}")

## 1c. XBRL facts

In [ ]:
from stance.seed import FACT_ALIASES

element_names = list({alias[1] for alias in FACT_ALIASES})
total_facts_inserted = 0

for company in WATCHLIST:
    cik = company["cik"]
    extracted = facts.extract_facts(company_facts_cache[cik], cik, element_names, FISCAL_YEARS)
    inserted = facts.store_facts(engine, extracted)
    total_facts_inserted += inserted
    print(f"{company['ticker']}: extracted {len(extracted)}, inserted {inserted}")

print(f"\nTotal facts inserted: {total_facts_inserted}")

## Exit criteria check

In [ ]:
with engine.connect() as conn:
    n_chunks = conn.execute(text("SELECT COUNT(*) FROM chunks")).scalar()
    n_facts = conn.execute(text("SELECT COUNT(*) FROM xbrl_facts")).scalar()
    n_filings = conn.execute(text("SELECT COUNT(*) FROM filings")).scalar()
print(f"filings={n_filings}, chunks={n_chunks}, xbrl_facts={n_facts}")
assert n_chunks > 0 and n_facts > 0, "Ingestion produced no data — check acquire/parse steps above."

# Hand-check: Apple's FY24 revenue should be a 12-digit number in the ~380-400B range.
from stance.numeric.resolver import resolve_fact
aapl_rev = resolve_fact(engine, "revenue", "0000320193", 2024)
print(f"AAPL FY24 revenue fact: {aapl_rev['value']} {aapl_rev['unit']} (fact_id={aapl_rev['fact_id']})")